# Exploratory Data Analysis (EDA) - 
  Olist E-Commerce Dataset

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set display options
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [ ]:
# Connect to SQLite database
conn = sqlite3.connect('../olist.db')

In [ ]:
# Get list of all tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

for table_name in tables['name']:
    df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 1", conn)
    count = pd.read_sql_query(f"SELECT COUNT(*) FROM {table_name}", conn).iloc[0, 0]
    print(f"Table: {table_name} | Rows: {count} | Columns: {len(df.columns)}")

## Revenue Overview

In [ ]:
# Monthly revenue trend plot
query = """
SELECT 
    strftime('%Y-%m', order_purchase_timestamp) AS month,
    SUM(payment_value) AS revenue
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status != 'canceled'
GROUP BY 1
ORDER BY 1;
"""
try:
    df_revenue = pd.read_sql_query(query, conn)
    fig = px.bar(df_revenue, x='month', y='revenue', title='Monthly Revenue Trend')
    fig.show()
except Exception as e:
    print("Could not execute query. Ensure schema is loaded.", e)

## Review Score Distribution

In [ ]:
# Histogram of review scores
query = "SELECT review_score, COUNT(*) as count FROM order_reviews GROUP BY 1 ORDER BY 1;"
try:
    df_reviews = pd.read_sql_query(query, conn)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df_reviews, x='review_score', y='count', palette='viridis')
    plt.title('Distribution of Review Scores')
    plt.xlabel('Review Score')
    plt.ylabel('Number of Reviews')
    plt.show()
except Exception as e:
    print("Could not execute query. Ensure schema is loaded.", e)

## Delivery Analysis

In [ ]:
# Scatter of delivery delay vs review score
query = """
SELECT 
    CAST(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) AS INTEGER) AS delay_days,
    review_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
LIMIT 10000;
"""
try:
    df_delivery = pd.read_sql_query(query, conn)
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_delivery, x='review_score', y='delay_days')
    plt.title('Impact of Delivery Delay on Review Score')
    plt.ylabel('Delay (Days) > 0 means late')
    plt.xlabel('Review Score')
    plt.ylim(-20, 20)
    plt.show()
except Exception as e:
    print("Could not execute query. Ensure schema is loaded.", e)

## RFM Preview

In [ ]:
# Quick RFM calculation and segment counts
query = """
WITH customer_rfm AS (
    SELECT 
        c.customer_unique_id,
        CAST(julianday('2018-10-01') - julianday(MAX(o.order_purchase_timestamp)) AS INTEGER) AS recency,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(p.payment_value) AS monetary
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_payments p ON o.order_id = p.order_id
    GROUP BY c.customer_unique_id
)
SELECT 
    CASE
        WHEN recency <= 30 AND frequency > 1 THEN 'Champions'
        WHEN recency > 180 AND frequency = 1 THEN 'Lost'
        WHEN recency > 90 AND frequency > 1 THEN 'At Risk'
        ELSE 'Other'
    END AS segment,
    COUNT(*) as customer_count
FROM customer_rfm
GROUP BY 1
ORDER BY 2 DESC;
"""
try:
    df_rfm = pd.read_sql_query(query, conn)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df_rfm, x='customer_count', y='segment', palette='mako')
    plt.title('Customer Segments (Preview)')
    plt.show()
except Exception as e:
    print("Could not execute query. Ensure schema is loaded.", e)

## Next Steps
- Complete the SQL scripts in `sql/`
- Run the data loading script
- Build the Streamlit dashboard